# Basic design of RF systems — B: electron storage ring

Current tutors: S. Albright, H. Damerau, A. Lasheen, M. Taquet

Contributors: J. Flowerdew, L. Intelisano, D. Quartullo, F. Tecker, C. Völlinger, L. Valle, M. Zampetakis

## Links

- Introductory CAS website: https://indico.cern.ch/event/1622828/
- Programme of the CAS: https://indico.cern.ch/event/1622828/attachments/3196386/5990113/Timetable_Introductory2026_ver1.6b.pdf
- Python software installation for transverse (and longitudinal) exercises: https://github.com/cerncas/hands-on-python/blob/main/Setup_Instructions.md
- Longitudinal hands-on, link to content and cheat sheets: https://indico.cern.ch/event/1622828/contributions/7202974/

# Introduction

This hands-on session exists in **two versions**, please choose **one** according to your interest:
- **Version A**: design the RF system for a proton synchrotron, the upgrade of the SPS to a hadron injector for the Future Circular Collider (FCC-hh)
- **Version B**: develop the RF system for a hypothetical beam energy and current upgrade of an electron storage ring (Soleil)

**This notebook is version B: electron storage ring.**

The *Lecture slides* lines below point to the slides of the **Longitudinal Dynamics** lecture (F. Tecker) and of the **RF Systems** lecture (C. Völlinger) at this school, where the corresponding topics are introduced. The numbers are the slide numbers printed at the bottom of the slides.

Each question is followed by an empty code cell for your calculation. The given values are defined once, in the cell below the parameter table.

## Import modules

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as sciCont

# Design of an RF system upgrade for an electron storage ring

### Design an RF system to run the Soleil electron storage ring at higher energy and beam current

In this hands-on session we will develop the RF system for a hypothetical beam energy and current upgrade for an electron storage ring (Soleil). As a storage ring the particle energy should just be kept constant.

The increase in beam energy would of course shift the critical photon energy, which would render several existing (infrared) beam lines useless. These specificities are intentionally ignored in the context of this exercise.

The goal of the session is to calculate the relevant longitudinal parameters.

The notebook is constructed with the following purpose in mind:

1. Start from the dominating energy loss per turn due to synchrotron radiation.
2. Knowing energy loss per turn and the number of particles stored the average RF power to the beam is calculated.
3. The RF frequency can be any integer multiple of the revolution frequency. Collect arguments for the choice of the RF frequency.
4. The energy transfer does not only take place from the cavity to the beam, but also the beam may induce significant voltage back into the cavity. This phenomenon has been introduced as beam loading.
5. Without the RF system, the beam quickly loses energy. It is interesting to estimate how many turns it would survive and how rapidly its energy decreases. Synchrotron radiation has the advantage of introducing damping and any oscillations of particles  reduced. The characteristic timescale of that process is the synchrotron radiation damping time.
6. New RF systems for particle accelerators are rarely designed from scratch, but inspired by existing installations. Compare your RF system design with the existing one of Soleil, as well as with the RF system of the larger ESRF main ring.

Key upgrade parameters:
- Higher energy: $3.5\,\mathrm{GeV}$ instead of $2.75\,\mathrm{GeV}$.
- Higher beam current: $800\,\mathrm{mA}$ instead of $500\,\mathrm{mA}$.
- Bunch spacing of $25\,\mathrm{ns}$.
- Design the new RF system which can work in combination with the existing one.

**Basic parameters of the Soleil electron storage ring ([parameter table](https://www.synchrotron-soleil.fr/en/research/sources-and-accelerators/parameters-accelerators-storage-ring))**

| Parameter             |                                                                   |
| --------------------- | ----------------------------------------------------------------- |
| Beam energy           | $E = 2.75\,\mathrm{GeV}\rightarrow 3.5\,\mathrm{GeV}$             |
| Beam current          | $I_\mathrm{b} = 500\,\mathrm{mA} \rightarrow 800\,\mathrm{mA}$    |
| Circumference         | $2 \pi R = 354.097\,\mathrm{m}$                                   |
| Bending radius        | $\rho = 5.36\,\mathrm{m}$                                         |
| Phase slip factor and momentum compaction | $\eta = 1/\gamma^2_\mathrm{tr} - 1/\gamma^2 \simeq 1/\gamma^2_\mathrm{tr} = \alpha = 4.16 \cdot 10^{-4}$ |
| Harmonic of RF system | $h = 416$                                                         |
| RF frequency          | $f_\mathrm{RF} = 352.2\,\mathrm{MHz}$                             |

In [ ]:
# Physical constants
c0 = sciCont.c  # m/s
e0 = sciCont.e  # C
epsilon0 = sciCont.epsilon_0  # F/m
electronMass = sciCont.physical_constants["electron mass energy equivalent in MeV"][0] * 1e6  # eV
electronCharge = 1  # e

# Given values, from the table above
beamEnergyBefore = 2.75e9  # eV
beamEnergyAfter = 3.5e9  # eV
circumference = 354.097
bendingRadius = 5.36  # m
momentumCompactionFactor = 4.16e-4

## Exercise 1: Average energy loss per turn

*Lecture slides: Tecker 44*

1. Calculate the average energy loss per turn to be restituted before and after the upgrade.
    - *Consider only synchrotron radiation generated in the main bending magnets: $\Delta E_\mathrm{turn} = e^2 \gamma^4/(3 \varepsilon_0 \rho)$, where $\varepsilon_0 \simeq 8.85 \cdot 10^{-12}\,\mathrm{F/m}$* *(Tecker 44)*
    - *In a real synchrotron light source a non-negligible fraction of the beam energy loss per turn is caused by insertion devices (undulators, wigglers). In Soleil at $2.75\,\mathrm{GeV}$, this amounts to an additional energy loss about $205\,\mathrm{keV}$ per turn.*
    - *With SI constants this formula gives the energy loss in joules: divide by the elementary charge to get it in eV.*

In [ ]:
# --- Solution ---
energyLossPerTurnBefore = (
    e0**2 * (beamEnergyBefore / electronMass) ** 4 / (3 * epsilon0 * bendingRadius) / e0
)

print("Energy loss per turn before upgrade: " + str(energyLossPerTurnBefore / 1e3) + " keV")
energyLossPerTurnAfter = (
    e0**2 * (beamEnergyAfter / electronMass) ** 4 / (3 * epsilon0 * bendingRadius) / e0
)

print("Energy loss per turn after upgrade: " + str(energyLossPerTurnAfter / 1e3) + " keV")

2. Plot the energy loss versus beam energy.

In [ ]:
# --- Solution ---
beamEnergyRange = np.linspace(beamEnergyBefore, beamEnergyAfter, 100)
energyLossPerTurn = (
    e0**2 * (beamEnergyRange / electronMass) ** 4 / (3 * epsilon0 * bendingRadius) / e0
)

plt.figure()
plt.plot(beamEnergyRange / 1e9, energyLossPerTurn / 1e3)
plt.xlabel(r"$E$ [GeV]")
plt.ylabel(r"$\Delta E/$turn [keV]")
plt.show()

3. Optionally, calculate the approximate radiation damping times of the synchrotron oscillations at $2.75\,\mathrm{GeV}$ and $3.5\,\mathrm{GeV}$. *(Tecker 44, and the Electron Beam Dynamics lecture)*
    - *The damping time scales with the revolution period, $T_\mathrm{rev} = C/(\beta c)$.*

In [ ]:
# --- Solution ---
revolutionTime = circumference / c0
dampingIntegralD = momentumCompactionFactor * circumference / (2 * np.pi * bendingRadius)

print("D = " + str(dampingIntegralD) + " (well below one, 2+D can be approximate to 2)")

dampingTimeBefore = (
    beamEnergyBefore * revolutionTime / energyLossPerTurnBefore
)  # not radiationPowerBefore (which is for the whole beam)
dampingTimeAfter = (
    beamEnergyAfter * revolutionTime / energyLossPerTurnAfter
)  # not radiationPowerAfter

print("SR Damping time before: " + str(dampingTimeBefore * 1e3) + " ms")
print("SR Damping time after:  " + str(dampingTimeAfter * 1e3) + " ms")

## Exercise 2: Average RF power

1. What is the average power to the beam before and after the upgrade?
    - *Average means over a turn.*
    - *Calculate the revolution frequency and period.*
    - *The power loss is defined by the energy lost per time. The number of charges can of course be determined from the beam current.*
    - *Equivalently: the power is the energy loss per turn expressed in volts, times the beam current in amperes.*

In [ ]:
# --- Solution ---
# Electrons therefore beta = 1
revolutionFrequency = c0 / circumference

print("Revolution frequency: " + str(revolutionFrequency / 1e3) + " kHz")
beamCurrent = 500e-3  # A
nElectrons = (beamCurrent / e0) / revolutionFrequency
radiationPowerBefore = energyLossPerTurnBefore * revolutionFrequency * nElectrons * e0

print("Average power to beam before upgrade: " + str(radiationPowerBefore / 1e3) + " kW")
beamCurrent = 800e-3  # A
nElectrons = (beamCurrent / e0) / revolutionFrequency
radiationPowerAfter = energyLossPerTurnAfter * revolutionFrequency * nElectrons * e0

print("Average power to beam after upgrade: " + str(radiationPowerAfter / 1e3) + " kW")

2. Plot the required RF power versus beam energy.

In [ ]:
# --- Solution ---
beamEnergyRange = np.linspace(beamEnergyBefore, beamEnergyAfter, 100)
energyLossPerTurn = (
    e0**2 * (beamEnergyRange / electronMass) ** 4 / (3 * epsilon0 * bendingRadius) / e0
)

beamCurrent = 500e-3
radiationPowerBeforeCurrent = energyLossPerTurn * beamCurrent

beamCurrent = 800e-3
radiationPowerAfterCurrent = energyLossPerTurn * beamCurrent

plt.figure()
plt.plot(beamEnergyRange / 1e9, radiationPowerBeforeCurrent / 1e3)
plt.plot(beamEnergyRange / 1e9, radiationPowerAfterCurrent / 1e3)
plt.xlabel("$E$ [GeV]")
plt.ylabel("$P$ [kW]")
plt.show()

3. Why should the installed RF power actually be higher?

In [ ]:
# --- Solution ---
print("Increase of the radiated power: " + str(radiationPowerAfter / radiationPowerBefore))

# Answer: to compensate the radiated power, which rises from 0.47 MW to 2.0 MW: the beam current is
# 1.6 times higher and the energy loss per turn 2.6 times higher (it scales with E^4)

## Exercise 3: RF frequency and harmonic

*Lecture slides: Tecker 15, 43 · Völlinger 9–13*

1. The bunch spacing of $25\,\mathrm{ns}$ corresponds to a bunch frequency of about $40\,\mathrm{MHz}$. What is the corresponding (nearest) harmonic number?

In [ ]:
# --- Solution ---
minimumFrequency = 1 / 25e-9
harmonicNumber = minimumFrequency / revolutionFrequency
print("Minimum frequency: " + str(minimumFrequency / 1e6) + " MHz")
print("40 MHz/revolution frequency at extraction: " + str(harmonicNumber))
harmonicNumber = round(harmonicNumber)
print("Nearest integer harmonic number, h = " + str(harmonicNumber))

2. Calculate the exact RF frequency for that harmonic number.

In [ ]:
# --- Solution ---
exactFrequency = harmonicNumber * revolutionFrequency

print("RF frequency for h = " + str(harmonicNumber) + ": " + str(exactFrequency / 1e6) + " MHz")

3. Is it useful, seen that the present harmonic number is $h=416$?
    - *The upgraded RF system should operate at or at an integer multiple of existing RF frequency, hence $h= n \cdot 416$ with, $n=1,2,\dots$*

In [ ]:
# --- Solution ---
print(
    "The ratio "
    + str(416)
    + "/"
    + str(harmonicNumber)
    + " = "
    + str(416 / harmonicNumber)
    + " is unfavorable; 352.2 MHz is not dividable by 40 MHz."
)

4. For the upgrade, twice the present frequency is chosen ($h=832$, $f_\mathrm{RF} = 704$ MHz) to generate additional voltage more easily and with compact accelerating cavities. *(Völlinger 11, 13)* What is the spacing between buckets for $h=416$ and for $h=832$? How can a bunch spacing of about $25\,\mathrm{ns}$ be obtained? *(Tecker 15, 43)*

In [ ]:
# --- Solution ---
chosenHarmonic = 2 * 416
chosenFrequency = chosenHarmonic * revolutionFrequency
bucketSpacing = 1 / chosenFrequency

print("Bucket spacing: " + str(bucketSpacing * 1e9) + " ns")

print("Bucket spacing for h = 416: " + str(1 / (416 * revolutionFrequency) * 1e9) + " ns")
print("25 ns in units of buckets at h = 416: " + str(25e-9 * 416 * revolutionFrequency))

# Answer: 2.84 ns at h = 416 and 1.42 ns at h = 832. A spacing of 25 ns is 8.8 buckets at h = 416, not an
# integer: the closest filling pattern is one bunch every 9th bucket, i.e. a spacing of 25.6 ns

## Exercise 4: Requirements for beam loading: beam induced voltage and power

*Lecture slides: Völlinger 72–73, 77–79*

1. Assume an RF cavity resonator with an $R/Q$ of about $44\,\mathrm{\Omega}$. *(Völlinger 73, 79)* What is the beam induced voltage and power due to the passage of one bunch? *(Völlinger 78)* How does the power compare to the power lost by synchrotron radiation?
    - *For the sake of the exercise, we assume here that the whole beam is concentrated in one bunch.*
    - *The voltage induced by a charge passing the cavity is given in the cheat sheet.*
    - *The power is the induced voltage times the beam current, and the beam current is the total charge times the revolution frequency.*

In [ ]:
# --- Solution ---
RUponQ = 44
VInduced = nElectrons * e0 * RUponQ * chosenHarmonic * revolutionFrequency * 2 * np.pi

print("Beam loading induced voltage: " + str(VInduced / 1e3) + " kV")

beamCurrent = 800e-3  # A
beamLoadingPower = VInduced * beamCurrent

print("Beam loading power (flat top): " + str(beamLoadingPower / 1e3) + " kW")

print(
    "Ratio to the synchrotron radiation power after upgrade: "
    + str(beamLoadingPower / radiationPowerAfter)
)

# Answer: about 150 kW, NB: this assumes single bunch passage

2. Under which circumstances do you really need that power?

In [ ]:
# --- Solution ---
# Answer: this would only be needed to fully compensate beam loading
# Detune RF cavities such that the system of power generator and beam is resonant at $832 f_\mathrm{rev}$
# In steady state, when the decay of the voltage balances the induced one, a cavity resonant at the RF
# frequency reaches V = 2 I_beam R = 2 I_beam Q (R/Q) (short bunches, circuit definition of R/Q),
# i.e. Q / (pi h) times the single-passage value

## Exercise 5: Beam life time with no RF

*Lecture slides: Tecker 44, 53*

1. Without RF, the particles lose energy at every turn and are lost when they leave the momentum acceptance, which is on the order of 0.5%. At which energy are the particles lost, before and after the upgrade? *(Tecker 53)*
    - *Convert the momentum acceptance into a momentum, then into an energy.*

In [ ]:
# --- Solution ---
momentumRatioAcceptance = 0.5e-2

beamEnergy = 2.75e9  # eV
designMomentum = np.sqrt(beamEnergy**2 - electronMass**2)
lossMomentumBefore = designMomentum * (1 - momentumRatioAcceptance)
lossEnergyBefore = np.sqrt(lossMomentumBefore**2 + electronMass**2)

print(
    "Before upgrade, momentum at which particles are lost: "
    + str(lossMomentumBefore / 1e9)
    + " GeV/c"
)
print(
    "Before upgrade, energy at which particles are lost:   " + str(lossEnergyBefore / 1e9) + " GeV"
)

beamEnergy = 3.5e9  # eV
designMomentum = np.sqrt(beamEnergy**2 - electronMass**2)
lossMomentumAfter = designMomentum * (1 - momentumRatioAcceptance)
lossEnergyAfter = np.sqrt(lossMomentumAfter**2 + electronMass**2)

print(
    "After upgrade, momentum at which particles are lost: "
    + str(lossMomentumAfter / 1e9)
    + " GeV/c"
)
print("After upgrade, energy at which particles are lost:   " + str(lossEnergyAfter / 1e9) + " GeV")

2. How many turns would the beam survive without RF, and how long is that? For a first estimate assume that the energy loss per turn stays constant.
    - *Divide the energy window by the energy loss per turn of Exercise 1, and convert the number of turns into a time with the revolution frequency.*

In [ ]:
# --- Solution ---
nTurnsUntilLostBefore = (beamEnergyBefore - lossEnergyBefore) / energyLossPerTurnBefore
lifeTimeBefore = nTurnsUntilLostBefore / revolutionFrequency

print(
    "Life time (before upgrade): "
    + str(lifeTimeBefore * 1e6)
    + " us ("
    + str(nTurnsUntilLostBefore)
    + " turns)"
)

nTurnsUntilLostAfter = (beamEnergyAfter - lossEnergyAfter) / energyLossPerTurnAfter
lifeTimeAfter = nTurnsUntilLostAfter / revolutionFrequency

print(
    "Life time (after upgrade):  "
    + str(lifeTimeAfter * 1e6)
    + " us ("
    + str(nTurnsUntilLostAfter)
    + " turns)"
)

3. Optionally, take into account that the energy loss per turn decreases as the beam loses energy: compute the beam energy turn after turn, and plot it together with the constant-loss estimate.
    - *Loop over the turns, recomputing the energy loss per turn from the current beam energy.*

In [ ]:
# --- Solution ---
beamEnergyInitial = 3.5e9
nTurns = 100
turnRange = np.array(range(nTurns))
energyTurnByTurn = np.zeros(nTurns)
energyLossPerTurnInitial = (
    e0**2 * (beamEnergyInitial / electronMass) ** 4 / (3 * epsilon0 * bendingRadius) / e0
)

beamEnergy = beamEnergyInitial
for turn in turnRange:
    energyTurnByTurn[turn] = beamEnergy
    energyLossPerTurn = (
        e0**2 * (beamEnergy / electronMass) ** 4 / (3 * epsilon0 * bendingRadius) / e0
    )
    beamEnergy = beamEnergy - energyLossPerTurn

plt.figure()
plt.plot(turnRange, energyTurnByTurn / 1e9)
plt.plot(turnRange, (beamEnergyInitial - turnRange * energyLossPerTurnInitial) / 1e9)
plt.xlabel("Turn number")
plt.ylabel("Beam energy [GeV]")
plt.show()

# Answer: the loss per turn decreases with the energy, so the beam survives a bit longer than the constant-loss estimate

## Exercise 6: Comparison with RF system at ESRF

*Lecture slides: Völlinger 13*

1. Compare the parameters of the (additional) RF system with the one of the storage ring at ESRF. Exchange with the tutors on how the different RF systems compare with one another.

| Parameter                       | Unit | ESRF  | Soleil | Soleil (CAS upgrade) |
| ------------------------------- | ---- | ----- | ------ | -------------------- |
| Beam energy, $E$                | GeV  | 6     | 2.75   | 3.5                  |
| Beam current, $I_\mathrm{beam}$ | mA   | 200   | 500    | 800                  |
| Circumference, $2 \pi R$        | m    | 844   | 354    | 354                  |
| Bending radius, $\rho$          | m    | 23.37 | 5.36   | 5.36                 |
| RF harmonic, $h$                |      | 992   | 416    | 416 and 832          |
| RF frequency, $f_\mathrm{RF}$   | MHz  | 352   | 352    | 352 and 704          |
| RF voltage, $V_\mathrm{RF}$     | MV   | 6.5   | 3      | 3 + >1.5             |
| Number of cavities              |      | 14    | 4      | 4 + ~4               |